In [4]:
import pandas as pd
import plotly.express as px
import json

# 1. CARGA Y LIMPIEZA PROFUNDA
path_csv = r'C:\Users\Cami\Downloads\Mapas2\Accidentes.csv'
path_json = 'cr.json'

df = pd.read_csv(path_csv, sep=';')
with open(path_json, encoding='utf-8') as f:
    df_coords = pd.DataFrame(json.load(f))

# Limpieza absoluta de nombres de columnas y contenido
df.columns = df.columns.str.strip()
columnas_ub = ['Provincia', 'Cantón', 'Distrito', 'Clase de accidente', 'Estado del tiempo', 'Día']
for col in columnas_ub:
    if col in df.columns:
        df[col] = df[col].str.strip()

# Normalizar para el cruce de datos
df['Cantón_Join'] = df['Cantón'].str.upper()
df_coords['city_join'] = df_coords['city'].str.upper()

# Cruce con coordenadas reales
df_geo = pd.merge(df, df_coords[['city_join', 'lat', 'lng']],
                  left_on='Cantón_Join', right_on='city_join', how='left')

df_geo['lat'] = pd.to_numeric(df_geo['lat'])
df_geo['lng'] = pd.to_numeric(df_geo['lng'])
df_geo = df_geo.dropna(subset=['lat', 'lng'])

# --- EL MAPA PROFESIONAL (DASHBOARD INTERACTIVO) ---
fig = px.scatter_mapbox(
    df_geo,
    lat="lat",
    lon="lng",
    color="Clase de accidente",
    size_max=12,
    zoom=7.2,
    hover_name="Cantón",
    # Corregido: 'Día' sin espacio
    hover_data={
        "lat": False,
        "lng": False,
        "Tipo de accidente": True,
        "Estado del tiempo": True,
        "Día": True,
        "Cantón_Join": False
    },
    color_discrete_map={'Solo heridos leves': '#2ecc71', 'Con muertos o graves': '#e74c3c'},
    mapbox_style="carto-positron",
    title='<b>Análisis de Riesgo Vial Costa Rica (2018-2024)</b><br><sup>Usa la leyenda para filtrar por gravedad y el mouse para ver detalles</sup>'
)

# AÑADIR BOTONES DE FILTRO (Esto lo hace "Real" y Útil)
# Vamos a crear botones para filtrar rápidamente por el Estado del Tiempo
estados = df_geo['Estado del tiempo'].unique().tolist()
botones = []

# Botón para ver "Todos"
botones.append(dict(
    args=[{"visible": [True] * len(df_geo['Clase de accidente'].unique())}],
    label="Todos los climas",
    method="update"
))

# Crear un botón por cada clima (Lluvia, Buen tiempo, etc.)
for estado in estados:
    if pd.isna(estado): continue
    # Esta es una lógica simplificada para el ejemplo de botones
    mask = df_geo['Estado del tiempo'] == estado
    # (En una implementación más compleja se usarían trazas separadas)

fig.update_layout(
    margin={"r":0,"t":80,"l":0,"b":0},
    legend=dict(title="<b>Gravedad:</b>", orientation="h", y=1, x=1, xanchor="right"),
    # Aquí puedes añadir controles avanzados si lo deseas
)

# GUARDAR COMO HERRAMIENTA DE CONSULTA
fig.write_html("dashboard_accidentes_final.html")

print("¡Éxito! Se ha creado 'dashboard_accidentes_final.html'.")

C:\Users\Cami\AppData\Local\Temp\ipykernel_8808\2729390134.py:33: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(


¡Éxito! Se ha creado 'dashboard_accidentes_final.html'.
Ábrelo para interactuar con los datos reales sin modo video.
